# [기초-실습] 통계 101×데이터 분석: (9-10장) 가설검정의 주의점/인과와 상관

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## ⚙️ 환경 준비

### 1단계 · 한글 폰트 설치

- 그래프에 한글이 깨지지 않도록 나눔 폰트를 설치합니다.

- 실행 후 **[런타임] - [세션 다시 시작]**을 한 번 눌러야 폰트가 적용됩니다.

In [ ]:
# 구글 코랩 환경에서 한글 폰트 설치 및 설정하기
# 필요시 아래 코드 실행 후, [런타임] - [세션 다시 시작] 후 셀을 다시 실행하세요.
!pip install statsmodels scikit-learn
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

### 2단계 · 라이브러리 불러오기

- 이번 실습에서 쓰는 도구입니다. **한 번만 실행**해 두면 끝까지 사용합니다.

- `TTestIndPower`는 검정력·표본크기를 계산하는 클래스, `NearestNeighbors`는 경향점수가 가장 가까운
  짝을 찾아주는 도구입니다.

In [ ]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.power import TTestIndPower
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

---

# 가설검정의 주의점

> p-값 하나로 결론을 내리면 무엇을 놓치는지, 그리고 "유의한 결과"가 어떻게 **만들어질 수 있는지**를 봅니다.

## 문제 1 · 펭귄 두 종의 몸무게는 정말 다를까?

`난이도 하` · `예상 20분`

**📖 상황**

- 7장에서 Adelie 수컷만 골라 회귀했을 때 p-값이 0.05를 넘어 **기각에 실패**했습니다.
  그때 우리는 "표본이 작아서"라고 정리했습니다.

- 이제 반대편에서 물어봅니다. **표본을 아주 크게 만들면 어떤 일이 벌어질까요?**

- 실제 팔머 기지 데이터에서 Adelie와 Chinstrap의 몸무게는 평균 **32g**밖에 차이나지 않습니다.
  펭귄 한 마리 몸무게(약 3,700g)의 **1%도 안 되는** 차이입니다.

- 이 32g을 붙잡고 표본만 늘려 가면, p-값은 어디까지 내려갈까요?

**🎯 이 문제로 배우는 것**

- **"표본크기 n이 커지면 p-값은 작아진다"**는 강의 9장의 명제를 실제 데이터로 확인하고,
  그래서 **효과크기(Effect Size)**를 함께 봐야 하는 이유를 숫자로 이해합니다.

In [ ]:
# 문제 1 · 데이터 준비 — 실행만 하세요
penguins = sns.load_dataset('penguins')
mass = penguins.dropna(subset=['body_mass_g', 'species'])

adelie    = mass.loc[mass['species'] == 'Adelie',    'body_mass_g']
chinstrap = mass.loc[mass['species'] == 'Chinstrap', 'body_mass_g']

print("Adelie    n=%3d   평균 %.1f g   표준편차 %.1f g" % (len(adelie),    adelie.mean(),    adelie.std(ddof=1)))
print("Chinstrap n=%3d   평균 %.1f g   표준편차 %.1f g" % (len(chinstrap), chinstrap.mean(), chinstrap.std(ddof=1)))

diff_g = adelie.mean() - chinstrap.mean()
print("\n두 종의 평균 차이: %.1f g  (Adelie 몸무게의 약 %.1f%%)" % (diff_g, abs(diff_g) / adelie.mean() * 100))

### Q1 · 두 종의 몸무게를 t-검정으로 비교해 봅시다

- `adelie`와 `chinstrap`의 몸무게에 대해 **이표본 t-검정(Two-sample t-Test)**을 수행하세요.
  5-6장 실습에서 쓴 그 검정입니다.

- 유의수준 0.05를 기준으로 **"기각한다 / 기각하지 못한다"까지 문장으로 출력**하세요.

- `💡 힌트` `stats.ttest_ind(x, y)`는 통계량과 p-값을 함께 돌려줍니다.

In [ ]:
# 문제 1 · Q1
# 여기에 코드를 작성해주세요.

t_stat, p_value = stats.ttest_ind(adelie, chinstrap)

print("=== Q1. 이표본 t-검정 ===")
print("t-통계량 = %.4f, p-값 = %.4f" % (t_stat, p_value))

alpha = 0.05
if p_value < alpha:
    print("p-값(%.4f) < 유의수준(0.05) → 귀무가설을 기각한다." % p_value)
else:
    print("p-값(%.4f) >= 유의수준(0.05) → 귀무가설을 기각하지 못한다." % p_value)

### Q2 · 효과크기(Cohen's d)를 직접 계산해 봅시다

- 강의 9장의 공식대로 `cohen_d(x, y)` **함수를 직접 만드세요. Q3에서 계속 씁니다.**
  - 분자: 두 집단 평균의 차이
  - 분모: 통합 표준편차 $s = \sqrt{\dfrac{(n_A-1)s_A^2 + (n_B-1)s_B^2}{n_A + n_B - 2}}$

- 만든 함수로 `adelie`와 `chinstrap`의 효과크기를 계산해 출력하세요.

- Cohen의 관례적 기준(0.2 작음 / 0.5 중간 / 0.8 큼)에서 어디에 놓이는지도 함께 적으세요.

- `💡 힌트` 표본표준편차는 `np.std(x, ddof=1)`, 표본 수는 `len(x)`입니다.

In [ ]:
# 문제 1 · Q2
# 여기에 코드를 작성해주세요.

def cohen_d(x, y):
    n_x, n_y = len(x), len(y)
    s_x, s_y = np.std(x, ddof=1), np.std(y, ddof=1)
    pooled_s = np.sqrt(((n_x - 1) * s_x**2 + (n_y - 1) * s_y**2) / (n_x + n_y - 2))
    return (np.mean(x) - np.mean(y)) / pooled_s

d = cohen_d(adelie, chinstrap)

print("\n=== Q2. Cohen's d ===")
print("Cohen's d = %.4f" % d)

abs_d = abs(d)
if abs_d < 0.2:
    size_label = "매우 작음"
elif abs_d < 0.5:
    size_label = "작음"
elif abs_d < 0.8:
    size_label = "중간"
else:
    size_label = "큼"
print("Cohen 관례 기준: '%s' 수준" % size_label)

### Q3 · 표본을 늘려 가며 p-값과 효과크기를 함께 추적해 봅시다

- Q1에서 본 두 종의 평균과 통합 표준편차를 **모집단 참값**으로 삼습니다.
  즉 "실제로 32g 차이가 나는 두 집단"에서 표본을 뽑는 상황을 만듭니다.

- `n = 30, 150, 1000, 5000, 20000` 각각에 대해, 두 집단에서 n마리씩 뽑아
  **t-검정과 Cohen's d를 400번 반복**하세요.

- 각 n마다 다음 세 값을 표로 정리해 출력하세요.
  **① p-값의 중위수 ② p < 0.05가 나온 비율(%) ③ Cohen's d의 평균**

- `💡 힌트` 모집단 참값은 이렇게 잡습니다.

  ```
  MU_A, MU_C = adelie.mean(), chinstrap.mean()
  SIGMA = np.sqrt(((len(adelie)-1)*adelie.std(ddof=1)**2 + (len(chinstrap)-1)*chinstrap.std(ddof=1)**2)
                  / (len(adelie) + len(chinstrap) - 2))
  ```

- `💡 힌트` 표본 추출은 `rng = np.random.default_rng(7)`로 시작해 `rng.normal(MU_A, SIGMA, n)`.
  결과는 리스트에 모아 `pd.DataFrame`으로 만들면 보기 좋습니다.

In [ ]:
# 문제 1 · Q3
# 여기에 코드를 작성해주세요.

MU_A, MU_C = adelie.mean(), chinstrap.mean()
SIGMA = np.sqrt(((len(adelie) - 1) * adelie.std(ddof=1)**2 +
                  (len(chinstrap) - 1) * chinstrap.std(ddof=1)**2)
                 / (len(adelie) + len(chinstrap) - 2))

print("\n=== Q3. 모집단 참값 ===")
print("MU_A = %.2f, MU_C = %.2f, SIGMA = %.2f" % (MU_A, MU_C, SIGMA))

n_list = [30, 150, 1000, 5000, 20000]
n_rep = 400
rng = np.random.default_rng(7)

results = []
for n in n_list:
    p_values = []
    d_values = []
    for _ in range(n_rep):
        sample_a = rng.normal(MU_A, SIGMA, n)
        sample_c = rng.normal(MU_C, SIGMA, n)
        _, p_val = stats.ttest_ind(sample_a, sample_c)
        d_val = cohen_d(sample_a, sample_c)
        p_values.append(p_val)
        d_values.append(d_val)

    p_values = np.array(p_values)
    d_values = np.array(d_values)

    results.append({
        "n": n,
        "p_median": np.median(p_values),
        "sig_rate": np.mean(p_values < 0.05) * 100,
        "d_mean": np.mean(d_values)
    })

df_result = pd.DataFrame(results)

print("\n=== Q3. 결과표 ===")
print("%6s %12s %14s %10s" % ("n", "p-값 중위수", "p<0.05 비율(%)", "d 평균"))
for row in results:
    print("%6d %12.4f %14.1f %10.4f" % (row["n"], row["p_median"], row["sig_rate"], row["d_mean"]))


### Q4 · 결과를 그래프로 그려 봅시다

- 가로축을 **표본크기 n**으로, 두 그림을 **나란히** 그리세요.
  - 왼쪽: n에 따른 **p < 0.05 비율(%)**
  - 오른쪽: n에 따른 **Cohen's d 평균**

- n이 30에서 20,000까지 넓게 퍼져 있으니 **가로축을 로그 스케일**로 두면 잘 보입니다.

- `💡 힌트` `fig, ax = plt.subplots(1, 2, figsize=(12, 4))` / `ax[0].set_xscale('log')`

In [ ]:
# 문제 1 · Q4
# 여기에 코드를 작성해주세요.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(df_result["n"], df_result["sig_rate"], marker='o', color='steelblue')
ax[0].set_xscale('log')
ax[0].set_xlabel("표본크기 n (log scale)")
ax[0].set_ylabel("p < 0.05 비율 (%)")
ax[0].set_title("표본크기에 따른 유의성 비율")
ax[0].axhline(5, color='gray', linestyle='--', linewidth=1)
ax[0].grid(alpha=0.3)

ax[1].plot(df_result["n"], df_result["d_mean"], marker='o', color='darkorange')
ax[1].set_xscale('log')
ax[1].set_xlabel("표본크기 n (log scale)")
ax[1].set_ylabel("Cohen's d 평균")
ax[1].set_title("표본크기에 따른 효과크기")
ax[1].axhline(0, color='gray', linestyle='--', linewidth=1)
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 💬 정리 · 결과를 말로 설명해 보기

- Q1에서 두 종의 몸무게는 통계적으로 유의한 차이가 **없었습니다.** 그런데 Q3에서 n=5,000일 때는
  95% 이상 유의하게 나왔습니다. **같은 32g인데 결론이 갈린 이유**는 무엇인가요?

- Q3의 표에서 n이 커질 때 **p-값**은 어떻게 변했나요? **Cohen's d**는 어떻게 변했나요?
  이 대비가 뜻하는 바를 한 문장으로 정리해 보세요.

- "통계적으로 유의하다"와 "실질적으로 의미 있다"는 같은 말인가요? 펭귄 32g을 예로 설명해 보세요.

- A/B 테스트에서 버튼 색을 바꿨더니 클릭률이 0.1%p 올랐고 p-값은 0.001이었습니다.
  이 결과만으로 버튼을 바꿔야 한다고 말할 수 있을까요?

- 논문이나 보고서에 **p-값만** 적는 것이 왜 부족한가요? 무엇을 함께 적어야 할까요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 강의 9장은 이렇게 적었습니다 — _"표본크기 𝓃이 커지면 𝑝값은 작아지므로 검출하고자 하는 효과크기를
  사전에 설정하고 표본크기 𝓃을 설계해야 합니다."_ Q3의 표가 이 문장을 그림으로 옮긴 것입니다.

- p-값은 **"차이가 있는가"**에 답하고, 효과크기는 **"그 차이가 얼마나 큰가"**에 답합니다.
  Q3의 표에서 어느 열이 어느 질문에 답하는지 나눠 보세요. 한 열은 n에 따라 움직이고, 다른 한 열은 꼼짝하지 않습니다.

- 32g은 Adelie 몸무게의 0.9%입니다. 저울의 눈금 하나 차이를 두고 "종에 따라 몸무게가 다르다"고
  보고서에 쓸 수 있을지 생각해 보세요.

- 마지막 질문의 답을 미국통계협회(ASA)는 2016년에 성명으로 내놓았습니다.
  요지는 "p-값 하나로 과학적 결론을 대신하지 말라"는 것이었습니다.

</details>

In [ ]:
# 문제 1 · 정리
# 여기에 의견을 작성해주세요.

# 같은 32g 차이라도 표본크기가 커지면 표준오차(SE)가 줄어들어 t-통계량이 커지고, 그만큼 작은 차이도 "우연이 아니다"라고 판단할 검정력이 커지기 때문입니다. 즉 차이의 크기(효과크기)는 그대로인데, n이 커지면서 그 차이를 감지하는 검정의 민감도만 높아진 것입니다. 표본이 원래 데이터(Adelie n=146, Chinstrap n=68)보다 훨씬 컸다면 처음부터 유의하게 나왔을 수도 있는 상황입니다.

# n이 커질수록 p-값은 계속 작아지고 유의 비율은 100%에 가까워지지만, Cohen's d는 약 32g/SIGMA 값 근처에서 거의 변하지 않고 일정하게 유지됩니다. 한 문장으로 정리하면, **"표본크기는 통계적 유의성(p-값)을 좌우하지만, 효과의 실질적 크기(Cohen's d)는 표본크기와 무관하게 일정하다"**는 것입니다.

# 같은 말이 아닙니다. 펭귄 32g 차이는 n=5,000에서는 p<0.05로 "통계적으로 유의"하지만, Cohen's d는 여전히 작거나 중간 수준에 머물러 있어 "실질적으로 큰 차이"라고 보기는 어렵습니다. 즉 통계적 유의성은 "이 차이가 우연히 생긴 것은 아니다"를 말해줄 뿐, "이 차이가 실제로 중요하다"는 것을 보장하지 않습니다.

# p-값 0.001은 이 0.1%p 차이가 우연일 가능성이 낮다는 뜻일 뿐, 그 차이가 비즈니스적으로 의미 있는 크기인지는 별개의 문제입니다. 트래픽이 매우 많은 서비스라면 표본이 커서 아주 작은 차이도 쉽게 유의하게 나오므로, 실제 매출/전환에 미치는 영향(효과크기), 구현 비용, 다른 지표에 미치는 부작용 등을 함께 봐야 결정할 수 있습니다. p-값만으로 "바꿔야 한다"고 말하기는 부족합니다.

# p-값은 표본크기에 크게 영향을 받기 때문에, n이 크면 사소한 차이도 유의하게 나오고 n이 작으면 큰 차이도 유의하지 않게 나올 수 있어 "차이의 크기"에 대한 정보를 주지 않습니다. 따라서 효과크기(Cohen's d 등), 신뢰구간, 표본크기를 함께 제시해야 독자가 "이 차이가 통계적으로 우연이 아니면서 동시에 실제로 얼마나 큰지"를 판단할 수 있습니다.


## 문제 2 · 그럼 몇 마리를 재야 할까?

`난이도 중` · `예상 25분`

**📖 상황**

- 문제 1은 "n을 늘리면 유의해진다"를 보여줬습니다. 뒤집어 읽으면 무서운 말입니다 —
  **n이 모자라면 진짜 있는 차이도 놓친다.**

- 진짜 차이가 있는데 놓치는 잘못을 **제2종 오류(β)**라 하고, 놓치지 않을 확률 **1 − β**를
  **검정력(Power)**이라 부릅니다. 관례적으로 80%를 목표로 삼습니다.

- 강의 9장은 이렇게 못박습니다 — _"미리 검출하고자 하는 효과크기를 정하고, 설정한 𝛼와 𝛽에 따라
  필요한 표본크기 𝓃을 결정해야 합니다."_

- 이번 문제는 그 계산을 **직접** 해 봅니다. 실험을 **시작하기 전에** 하는 계산입니다.

**🎯 이 문제로 배우는 것**

- 제1종·제2종 오류를 구분하고, **효과크기 → 필요 표본수**를 산출하는 검정력 분석을 익힙니다.
  문제 1의 시뮬레이션 결과가 이론값과 맞는지도 확인합니다.

In [ ]:
# 문제 2 · 데이터 준비 — 실행만 하세요
# 문제 1의 Q2·Q3를 먼저 완료해야 이 문제를 풀 수 있습니다.

analysis = TTestIndPower()   # 검정력 분석 도구

print("검정력 분석에 쓰이는 네 개의 값")
print("  ① 유의수준 alpha  — 제1종 오류를 허용하는 한계 (보통 0.05)")
print("  ② 검정력 power    — 1 - beta, 진짜 차이를 잡아낼 확률 (보통 0.80)")
print("  ③ 효과크기 d      — 검출하고자 하는 차이의 크기")
print("  ④ 표본크기 nobs1  — 집단당 표본 수")
print("\n→ 이 중 셋을 정하면 나머지 하나가 결정됩니다.")

### Q1 · 두 가지 오류를 표로 정리해 봅시다

- 강의 9장의 신약 예시를 씁니다. **H₀: "신약은 효과가 없다"**

- 아래 네 칸을 채우세요. 각 칸에 **오류의 이름**과 **무엇을 잃는가**를 함께 적으세요.

- 코드가 아니라 **말로** 채우는 문제입니다.

In [ ]:
# 문제 2 · Q1
#                          | H0를 기각함 (효과가 있다고 결론)  | H0를 기각 못함 (효과가 없다고 결론)
# 실제로 효과가 없을 때     | 제1종 오류 (α)— 효과 없는 약을 "효과 있다"고 잘못 판단   | 올바른 결론 (참 음성)
# 실제로 효과가 있을 때     |  올바른 결론 (참 양성)            |	제2종 오류 (β) — 효과 있는 약을 "효과 없다"고 놓침
#
# 제1종 오류(alpha)를 범하면 무엇을 잃나요?: 환자는 아무 효과 없는 약에 돈과 시간을 쓰고, 부작용 위험에 노출되며, 회사는 나중에 신뢰와 비용을 잃습니다.
# 제2종 오류(beta)를 범하면 무엇을 잃나요?: 환자들이 도움이 될 수 있었던 치료 기회를 놓치고, 회사는 유망한 신약을 사장시키는 손실을 입습니다. 즉 "있는 것을 없다고 놓치는" 대가를 치릅니다.
# 검정력(1-beta)을 한 문장으로 정의하면?: 검정력은 "실제로 효과가 있을 때, 그 효과를 통계적으로 올바르게 감지해낼 확률"입니다.

### Q2 · 문제 1의 시뮬레이션이 이론과 맞는지 확인해 봅시다

- 문제 1에서 구한 펭귄의 효과크기(d ≈ 0.074)에 대해, `n = 30, 150, 1000, 5000, 20000`
  각각의 **이론적 검정력**을 계산하세요.

- 이 값을 문제 1 Q3의 **'p < 0.05 비율'과 나란히 출력**해 두 값이 일치하는지 눈으로 확인하세요.

- `💡 힌트` `analysis.power(effect_size=..., nobs1=..., alpha=0.05, ratio=1)`
  — `effect_size`에는 효과크기의 **절대값**을 넣습니다.

In [ ]:
# 문제 2 · Q2
# 여기에 코드를 작성해주세요.

# 문제 1 Q2에서 구한 Cohen's d (약 0.074) 사용
effect_size = abs(d)   # 문제 1에서 계산한 cohen_d(adelie, chinstrap)

n_list = [30, 150, 1000, 5000, 20000]

print("=== Q2. 이론적 검정력 vs 시뮬레이션 유의 비율 ===")
print("Cohen's d = %.4f 기준\n" % effect_size)
print("%8s %14s %16s" % ("n", "이론적 검정력(%)", "시뮬레이션 비율(%)"))

for row in results:  # 문제 1 Q3의 results 리스트 재사용
    n = row["n"]
    power = analysis.power(effect_size=effect_size, nobs1=n, alpha=0.05, ratio=1)
    print("%8d %14.1f %16.1f" % (n, power * 100, row["sig_rate"]))

### Q3 · 80% 검정력에 필요한 표본수를 구해 봅시다

- 유의수준 0.05, 검정력 0.80을 목표로 할 때 **집단당 필요한 표본수**를 구하세요.

- 효과크기 네 가지에 대해 각각 계산해 표로 출력하세요.
  **① 펭귄의 d ≈ 0.074 ② 0.2(작음) ③ 0.5(중간) ④ 0.8(큼)**

- `💡 힌트` `analysis.solve_power(effect_size=..., alpha=0.05, power=0.8)`
  — 사람 수는 소수점이 될 수 없으니 `np.ceil()`로 올림하세요.

In [ ]:
# 문제 2 · Q3
# 여기에 코드를 작성해주세요.
effect_sizes = {
    "펭귄 d≈0.074": abs(d),   # 문제 1 Q2에서 계산한 값
    "작음 (0.2)": 0.2,
    "중간 (0.5)": 0.5,
    "큼 (0.8)": 0.8,
}

print("=== Q3. 80% 검정력에 필요한 집단당 표본수 (alpha=0.05) ===")
print("%16s %12s %14s" % ("효과크기", "d 값", "필요 n(집단당)"))

n_required = {}
for label, es in effect_sizes.items():
    n_needed = analysis.solve_power(effect_size=es, alpha=0.05, power=0.8)
    n_needed = int(np.ceil(n_needed))
    n_required[label] = n_needed
    print("%16s %12.3f %14d" % (label, es, n_needed))

### Q4 · 검정력 곡선을 그려 봅시다

- 효과크기를 **d = 0.5로 고정**하고, 집단당 표본수 n을 5부터 200까지 바꿔가며
  **검정력 곡선**을 그리세요.

- **80% 기준선**을 수평선으로 함께 그려, 곡선이 이 선을 넘는 지점이 Q3에서 구한 값과
  맞는지 확인하세요.

- `💡 힌트` `n_list = np.arange(5, 201, 5)` / `plt.axhline(0.8, color='red', linestyle='--')`

In [ ]:
# 문제 2 · Q4
# 여기에 코드를 작성해주세요.

n_range = np.arange(5, 201, 5)
power_curve = [analysis.power(effect_size=0.5, nobs1=n, alpha=0.05, ratio=1) for n in n_range]

plt.figure(figsize=(7, 4.5))
plt.plot(n_range, power_curve, marker='o', markersize=3, color='steelblue')
plt.axhline(0.8, color='red', linestyle='--', label='검정력 80% 기준선')
plt.axvline(n_required["중간 (0.5)"], color='gray', linestyle=':',
            label='Q3에서 구한 n=%d' % n_required["중간 (0.5)"])
plt.xlabel("집단당 표본수 n")
plt.ylabel("검정력 (Power)")
plt.title("검정력 곡선 (Cohen's d = 0.5)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



### 💬 정리 · 결과를 말로 설명해 보기

- Q3에서 펭귄의 32g 차이(d ≈ 0.074)를 80% 검정력으로 잡으려면 집단당 몇 마리가 필요했나요?
  팔머 기지 데이터는 **전체가 344마리**입니다. 이 연구는 애초에 가능했을까요?

- 검정력이 낮은 연구에서 "유의한 차이가 없었다"는 결과가 나왔을 때, 우리는 무엇을 알 수 있고
  **무엇을 알 수 없나요?** (7장 문제 4 Q5의 Adelie 수컷 회귀를 떠올려 보세요)

- 실험을 **시작하기 전에** 표본수를 정해야 하는 이유는 무엇인가요?
  데이터를 본 **뒤에** 정하면 무엇이 잘못될까요?

- 유의수준 alpha를 0.05에서 0.01로 낮추면 필요한 n은 늘어날까요, 줄어들까요?
  Q3의 코드를 고쳐 **직접 계산해** 확인해 보세요.

- "검출하고자 하는 효과크기를 얼마로 정할 것인가"는 **통계 문제**인가요, **도메인 문제**인가요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 네 개 값(alpha, beta, 효과크기, n) 중 **셋을 정하면 나머지 하나가 결정됩니다.**
  `solve_power`가 하는 일이 바로 이 방정식을 푸는 것입니다. 무엇을 미지수로 둘지 바꿔 보세요.

- **"유의한 차이 없음"은 "차이 없음"이 아닙니다.** 검정력이 20%인 연구라면, 진짜 차이가 있어도
  5번 중 4번은 놓칩니다. 그런 연구의 "차이 없음"은 증거라기보다 **정보 부족**입니다.

- 세 번째 질문의 답이 곧 다음 문제(p-해킹)의 출발점입니다. 표본수를 데이터를 보고 정한다는 것은
  곧 **결과를 보고 규칙을 바꾼다**는 뜻입니다.

- 마지막 질문 — 고혈압 치료제에서 "혈압을 몇 mmHg 낮추면 임상적으로 의미 있는가"를 정하는 사람은
  통계학자인가요, 의사인가요? 통계는 그 숫자를 **받아서** n을 계산해 줄 뿐입니다.

</details>

In [ ]:
# 문제 2 · 정리
# 여기에 의견을 작성해주세요.

# 계산해보면 집단당 약 2,900마리 안팎이 필요합니다(코드 실행 결과 확인). 팔머 펭귄 데이터는 전체 344마리, 그중 Adelie·Chinstrap 두 종만 합쳐도 훨씬 적기 때문에 이 연구는 애초에 32g이라는 작은 차이를 안정적으로 검출할 만큼 표본이 크지 않았습니다. 즉 "차이가 없다"는 결론이 아니라 "이 정도 표본으로는 있어도 못 찾는다"는 상황이었던 것입니다.
# 애초에 불가능.

# 알 수 있는 것은 "이 표본크기에서는 우연이 아니라고 말할 만큼 뚜렷한 차이는 못 찾았다"는 것뿐입니다. 
# 알 수 없는 것은 "실제로 차이가 전혀 없다"는 것입니다. 7장 Adelie 수컷 회귀에서처럼, 표본이 작고 검정력이 낮으면 실제로 존재하는 관계나 차이도 유의하지 않게 나올 수 있으므로, "유의하지 않다 = 효과가 없다"로 성급히 결론짓지 않아야 합니다.

# 데이터를 본 뒤 표본수나 분석 방법을 바꾸면(p-hacking, optional stopping) 우연히 유의한 결과가 나올 때까지 계속 표본을 추가하거나 검정을 반복하게 되어 제1종 오류율이 명목상 5%보다 훨씬 커집니다. 
# 사전에 목표 효과크기와 검정력을 정해 표본수를 고정해야 결과의 신뢰성이 보장됩니다.

# 필요한 n은 늘어납니다. alpha를 낮추면 "우연히 유의하다고 잘못 판단할 기준"이 더 엄격해지므로, 같은 효과를 그만큼 확실하게 검출하려면 더 많은 표본이 필요합니다(코드 실행 결과로 직접 확인 가능).

# 이는 도메인 문제입니다. 통계학은 주어진 효과크기·alpha·검정력에서 필요한 n을 계산해줄 뿐이고, "이 정도 차이면 실제로 의미가 있다"고 판단하는 기준(예: 신약이 몇 % 이상 효과가 있어야 임상적으로 유의미한지, 펭귄 32g이 생태학적으로 중요한지)은 해당 분야의 전문 지식과 가치 판단에서 나옵니다.

## 문제 3 · 유의한 결과는 '만들어질' 수 있다

`난이도 중` · `예상 25분`

**📖 상황**

- 심리학 분야의 과거 연구 100건을 재실험한 결과, 원래 유의했던 97건 중 **36건만** 다시
  유의했습니다(강의 9장). 이것이 **재현성 위기(Reproducibility Crisis)**입니다.

- 원인 중 하나는 **p-해킹** — 의도했든 아니든 p-값을 0.05 아래로 밀어 넣는 행위입니다.

- 이번 문제는 **아무 차이도 없는 두 집단**을 놓고, 분석 방식만 바꿔가며
  "유의한 결과"를 얼마나 만들어낼 수 있는지 **직접 세어 봅니다.**

- 진짜 차이가 없으므로 유의하다는 결론은 **전부 제1종 오류**입니다.
  절차를 지켰다면 5%여야 합니다.

**🎯 이 문제로 배우는 것**

- **정직한 분석 / 표본 추가형 / 다중비교형** 세 가지의 제1종 오류율을 직접 세어 비교하고,
  다중비교 보정이 왜 필요한지 확인합니다.

In [ ]:
# 문제 3 · 데이터 준비 — 실행만 하세요
N_SIM = 1000     # 시뮬레이션 반복 횟수
SEED  = 777      # 난수 시드 — Q1~Q4에서 각각 이 값으로 시작하면 몇 번 재실행해도 같은 결과가 나옵니다

print("이번 문제의 전제")
print("  두 집단 A, B는 모두 N(0, 1)에서 나옵니다 → 진짜 차이는 정확히 0")
print("  즉 귀무가설이 '참'인 상황이며, 유의하다는 결론은 모두 제1종 오류입니다.")
print("  절차를 지켰다면 오류율은 유의수준 5% 근처여야 합니다.")
print("\n각 조건마다 %d번씩 반복해 '유의하다고 결론 내린 비율'을 셉니다." % N_SIM)

### Q1 · 정직한 분석의 제1종 오류율을 확인해 봅시다

- **기준선**을 먼저 만듭니다. 각 집단에서 **20명씩 뽑아 딱 한 번만** t-검정하세요.

- 이것을 `N_SIM`번 반복해 **p < 0.05가 나온 비율**을 출력하세요.

- 이 값이 유의수준 0.05와 가까운지 확인하세요.

- `💡 힌트` 셀 맨 위에서 `rng = np.random.default_rng(SEED)`로 시작하세요.
  이렇게 하면 셀을 몇 번 다시 실행해도 같은 결과가 나옵니다. **Q2~Q4도 모두 이렇게 시작합니다.**

- `💡 힌트` `rng.normal(0, 1, 20)`으로 표본을 만들고, `for`문으로 반복해 세면 됩니다.

In [ ]:
# 문제 3 · Q1
# 여기에 코드를 작성해주세요.

rng = np.random.default_rng(SEED)

honest_sig = 0
for _ in range(N_SIM):
    group_a = rng.normal(0, 1, 20)
    group_b = rng.normal(0, 1, 20)
    _, p = stats.ttest_ind(group_a, group_b)
    if p < 0.05:
        honest_sig += 1

honest_rate = honest_sig / N_SIM * 100
print("=== Q1. 정직한 분석 ===")
print("p < 0.05 비율: %.2f%%  (기대값: 5%%)" % honest_rate)

### Q2 · 해킹 ① — 결과를 보고 표본을 더 모으면

- 강의 9장이 첫 번째로 꼽은 p-해킹입니다 — _"결과를 보며 표본크기를 늘려서는 안 됨"_

- 다음 규칙을 따르는 'p-해커'를 만드세요.
  1. 각 집단 **20명**으로 시작해 t-검정
  2. **p < 0.05면 "찾았다!"** 하고 즉시 멈춤
  3. **p ≥ 0.05면 각 집단에 10명씩 추가**해 다시 검정
  4. 이 과정을 **최대 5번**까지 반복

- `N_SIM`번 반복해 **한 번이라도 p < 0.05를 얻은 비율**을 출력하고, Q1의 값과 비교하세요.

- `💡 힌트` `rng = np.random.default_rng(SEED)`로 시작하세요.
  '해커 한 명'을 함수로 만들어 두면 반복이 깔끔해집니다.
  표본을 `list`로 두면 `.extend()`로 이어붙일 수 있습니다.

In [ ]:
# 문제 3 · Q2
# 여기에 코드를 작성해주세요.

def p_hacker_add_sample(rng, start_n=20, add_n=10, max_tries=5):
    """20명으로 시작, 유의할 때까지 최대 5번(각 10명씩 추가) 검정"""
    group_a = list(rng.normal(0, 1, start_n))
    group_b = list(rng.normal(0, 1, start_n))

    for _ in range(max_tries):
        _, p = stats.ttest_ind(group_a, group_b)
        if p < 0.05:
            return True   # "찾았다!"
        group_a.extend(rng.normal(0, 1, add_n))
        group_b.extend(rng.normal(0, 1, add_n))
    return False  # 5번 다 돌아도 유의하지 않음


rng = np.random.default_rng(SEED)

hack1_sig = 0
for _ in range(N_SIM):
    if p_hacker_add_sample(rng):
        hack1_sig += 1

hack1_rate = hack1_sig / N_SIM * 100
print("\n=== Q2. 해킹① (표본 추가형) ===")
print("p < 0.05를 한 번이라도 얻은 비율: %.2f%%" % hack1_rate)
print("Q1(정직한 분석) 대비 %.2f%%p 증가" % (hack1_rate - honest_rate))

### Q3 · 해킹 ② — 지표를 20개 재서 하나만 보고하면

- 강의 9장의 두 번째 p-해킹입니다 — _"마음에 드는 해석만 보고해서는 안 됨"_

- 이번엔 표본을 추가하지 않습니다. 대신 **아무 차이 없는 지표 20개**를 각각 20명씩 검정하고,
  **하나라도 유의하면 그것만 보고**합니다.

- `N_SIM`번 반복해 성공률을 출력하세요.

- 이론값 **1 − 0.95²⁰**과 나란히 출력해 비교하세요.

- `💡 힌트` `rng = np.random.default_rng(SEED)`로 시작하세요.
  바깥 `for`문은 시뮬레이션 반복, 안쪽 `for`문은 20개 지표입니다.
  `any()`를 쓰면 짧게 쓸 수 있습니다.

In [ ]:
# 문제 3 · Q3
# 여기에 코드를 작성해주세요.

rng = np.random.default_rng(SEED)

hack2_sig = 0
for _ in range(N_SIM):
    p_values = []
    for _ in range(20):
        group_a = rng.normal(0, 1, 20)
        group_b = rng.normal(0, 1, 20)
        _, p = stats.ttest_ind(group_a, group_b)
        p_values.append(p)
    if any(p < 0.05 for p in p_values):
        hack2_sig += 1

hack2_rate = hack2_sig / N_SIM * 100
theory_rate = (1 - 0.95**20) * 100

print("\n=== Q3. 해킹② (다중비교형) ===")
print("시뮬레이션 성공률: %.2f%%" % hack2_rate)
print("이론값 1 - 0.95^20 = %.2f%%" % theory_rate)


### Q4 · 세 결과를 나란히 놓고, 보정 효과까지 확인해 봅시다

- Q3의 방식에 **본페로니 보정**을 적용하면 오류율이 어떻게 되는지 계산하세요.
  (지표가 20개이므로 각 검정의 기준을 `0.05 / 20`으로 낮춥니다)

- Q1 ~ Q3 결과와 보정 후 결과, 총 **네 개를 막대그래프**로 그리세요.

- **α = 5% 기준선**을 수평선으로 함께 표시해 어느 것이 선을 넘는지 한눈에 보이게 하세요.

- `💡 힌트` 보정 후 계산도 `rng = np.random.default_rng(SEED)`로 시작하세요.
  Q3의 코드에서 기준값만 `0.05 / 20`으로 바꾸면 됩니다.

- `💡 힌트` `plt.bar(라벨리스트, 값리스트)` / `plt.axhline(5, color='red', linestyle='--')`

In [ ]:
# 문제 3 · Q4
# 여기에 코드를 작성해주세요.

rng = np.random.default_rng(SEED)

alpha_corrected = 0.05 / 20
bonf_sig = 0
for _ in range(N_SIM):
    p_values = []
    for _ in range(20):
        group_a = rng.normal(0, 1, 20)
        group_b = rng.normal(0, 1, 20)
        _, p = stats.ttest_ind(group_a, group_b)
        p_values.append(p)
    if any(p < alpha_corrected for p in p_values):
        bonf_sig += 1

bonf_rate = bonf_sig / N_SIM * 100

print("\n=== Q4. 본페로니 보정 적용 ===")
print("보정 기준(alpha=0.05/20=%.4f) 사용 시 성공률: %.2f%%" % (alpha_corrected, bonf_rate))

labels = ["Q1\n정직한 분석", "Q2\n표본 추가형", "Q3\n다중비교형", "Q3+보정\n(본페로니)"]
values = [honest_rate, hack1_rate, hack2_rate, bonf_rate]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, values, color=['steelblue', 'darkorange', 'firebrick', 'seagreen'])
plt.axhline(5, color='red', linestyle='--', label='α = 5% 기준선')
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, val + 1, "%.1f%%" % val,
              ha='center', fontsize=10)
plt.ylabel("제1종 오류율 (%)")
plt.title("정직한 분석 vs p-해킹 두 종류 vs 보정 후")
plt.legend()
plt.tight_layout()
plt.show()

### 💬 정리 · 결과를 말로 설명해 보기

- Q1의 값은 5%에 가까웠습니다. 이것이 뜻하는 바는 무엇인가요?
  **가설검정은 원래 무엇을 보장해 주는 도구**인가요?

- Q2와 Q3에서 오류율이 뛴 이유를 **각각** 설명해 보세요. 두 해킹의 메커니즘은 같나요, 다른가요?

- Q3의 결과는 무엇을 뜻하나요? "지표 20개를 재고 유의한 하나만 보고한다"는 연구를
  여러분은 신뢰할 수 있나요?

- 강의에 나온 **HARKing**(결과를 본 뒤에 가설을 만드는 행위)은 Q2·Q3 중 어느 쪽과 닮아 있나요?

- **사전 등록(Preregistration)**은 Q2와 Q3를 각각 어떻게 막아 주나요?
  두 해킹에 대해 따로 설명해 보세요.

- 문제 2에서 배운 '표본수 사전 설계'는 Q2를 막는 데 어떤 역할을 하나요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 가설검정이 보장하는 것은 **"정해진 절차를 지켰을 때 제1종 오류가 α 이하"**입니다.
  Q1은 그 보장이 지켜지는 모습이고, Q2·Q3는 절차를 어겼을 때 보장이 무너지는 모습입니다.
  같은 t-검정 함수를 썼는데 결과가 갈렸다는 점에 주목하세요 — 문제는 도구가 아니라 **쓰는 방식**입니다.

- Q2는 **같은 가설을 여러 번 물어본** 것이고, Q3는 **여러 가설을 물어보고 하나만 골라 보고한**
  것입니다. 표현은 다르지만 둘 다 "우연에게 기회를 여러 번 준" 셈입니다.

- 동전 던지기로 생각해 보세요. 앞면이 나올 확률은 1/2이지만, **20번 던져 한 번이라도**
  앞면이 나올 확률은 거의 1입니다. Q3의 숫자가 이 계산입니다.

- 강의 9장은 **가설검증형 연구 vs 탐색형 연구**를 구분합니다. Q3처럼 지표 20개를 훑는 일 자체가
  잘못은 아닙니다. 잘못은 그것을 **가설검증형인 척 보고**하는 데 있습니다.
  탐색에서 찾은 것은 **새 데이터로 다시 확인**해야 가설검증이 됩니다.

- p-값의 한계를 다루는 또 다른 접근으로 **베이즈 인수(Bayes Factor)**가 있습니다.
  강의 9장이 "11장 학습 후 진행 예정"으로 남겨 둔 주제입니다.

</details>

In [ ]:
# 문제 3 · 정리
# 여기에 의견을 작성해주세요.

# 정직하게 한 번만 검정했을 때 5%에 가까운 비율이 나온 것은, "진짜 차이가 없을 때 우연히 유의하다고 잘못 판단할 확률을 정확히 alpha 수준으로 통제한다"는 가설검정 본연의 약속이 지켜졌다는 뜻입니다. 가설검정은 "진실을 알려주는 도구"가 아니라 "정해진 절차를 지켰을 때 제1종 오류율을 그 값 이하로 묶어주는 도구"입니다.

# 두 해킹 모두 "검정을 여러 번 반복하고 그중 유리한 결과 하나를 고른다"는 점에서 근본 메커니즘은 같습니다. 다만 반복하는 대상이 다릅니다. Q2는 같은 가설을 표본을 늘려가며 반복 검정해 "유의해질 때까지 기다리는" 방식이고, Q3는 서로 다른 20개 지표를 병렬로 검정해 "그중 하나라도 유의한 것을 고르는" 방식입니다. 즉 순차적 반복 vs 동시다발적 반복이라는 형태만 다를 뿐, "여러 번 찔러보고 우연히 걸린 것을 고른다"는 본질은 동일합니다.

# 20개 중 하나만 유의하다고 보고하면, 시뮬레이션 성공률이 이론값 1-0.95²⁰(약 64%)에 근접하게 나옵니다. 즉 아무 차이도 없는 상황에서도 유의한 결과 하나를 "우연히" 얻을 확률이 60%가 넘습니다. 이런 식으로 "지표 20개를 재고 유의한 하나만 보고한다"는 연구는 그 결과가 진짜 효과인지 우연인지 구별할 수 없으므로 신뢰할 수 없습니다.

# HARKing(결과를 본 뒤 가설을 만드는 행위)은 **Q3(다중비교형)**과 더 닮아 있습니다. 여러 지표/변수를 미리 계획 없이 살펴본 뒤, 그중 우연히 유의하게 나온 하나를 골라 "원래 이걸 보려고 했다"는 듯이 가설을 사후적으로 짜맞추는 것이기 때문입니다.

# 사전 등록(Preregistration)이 Q2와 Q3를 막는 방식
# Q2(표본 추가형): 실험 전에 표본크기(n)와 중단 규칙을 미리 등록해두면, 결과를 보고 표본을 추가하거나 중간에 멈추는 행위 자체가 원천 차단됩니다. "언제까지, 몇 명까지 모을지"가 데이터를 보기 전에 고정되기 때문입니다.
# Q3(다중비교형): 실험 전에 "어떤 지표를 볼 것인지, 몇 개를 검정할 것인지, 다중비교 보정을 어떻게 할 것인지"를 미리 명시해두면, 나중에 20개 중 유리한 것 하나만 골라 보고하는 것이 불가능해집니다. 보고 전에 이미 "무엇을 볼지"가 확정되어 있기 때문입니다.

# 문제 2에서 배운 "목표 검정력에 맞춰 실험 전에 표본수를 미리 계산해 고정하는 것"은 Q2의 p-해킹을 원천적으로 차단합니다. 표본수가 이미 정해져 있으므로 "결과가 유의하지 않으니 더 모아보자"는 선택지 자체가 없어지고, 검정도 정확히 한 번만 이루어져 제1종 오류율이 alpha 수준으로 유지됩니다.

---

# 인과와 상관

> 상관을 인과로 착각하게 만드는 것은 무엇이며, 그것을 어떻게 걷어내는지를 봅니다.

## 문제 4 · 교란요인을 지우는 두 가지 방법 — 통제와 무작위화

`난이도 중` · `예상 30분`

**📖 상황**

- 7장 문제 1에서 이상한 것을 봤습니다. 펭귄 부리는 **전체로 보면 길수록 얇은데,
  종별로 나눠 보면 길수록 두꺼웠습니다.** 그때 "이 현상에는 이름이 붙어 있고,
  다음 시간 인과추론에서 다시 만난다"고 적어 두었습니다. **이제 그 이름을 붙입니다.**

- 원인은 **중첩요인(교란변수, Confounder)** — '종'이 부리 길이와 두께 양쪽에 영향을 주고
  있었습니다. 여기서 생겨난 가짜 음의 관계를 **허위상관(Spurious Correlation)**이라 부릅니다.

- 교란요인을 지우는 길은 두 갈래입니다.
  - **① 통제** — 층별로 나눠 보거나, 회귀식에 함께 넣기 (강의 10.3)
  - **② 무작위화** — 애초에 동전을 던져 배정하기, 즉 **무작위 통제 실험(RCT)** (강의 10.2)

- 이번 문제는 두 방법을 나란히 써 보고, **왜 무작위화가 더 강력한지** 확인합니다.

**🎯 이 문제로 배우는 것**

- 실제 데이터로 허위상관을 진단하고 통제로 걷어낸 뒤, 시뮬레이션으로 **무작위 배정의 힘**을
  확인합니다. 그리고 두 방법의 **결정적 차이**를 이해합니다.

In [ ]:
# 문제 4 · 데이터 준비 (1/2) — 실행만 하세요
# 펭귄 부리 데이터: 7장 문제 1에서 봤던 그 데이터입니다.
bills = penguins.dropna(subset=['bill_length_mm', 'bill_depth_mm', 'species'])

print("분석에 사용할 데이터: %d 행\n" % len(bills))
print("종별 부리 평균 —— '종'이 두 변수 모두에 영향을 주고 있다는 단서")
print(bills.groupby('species')[['bill_length_mm', 'bill_depth_mm']].mean().round(2))

### Q1 · 전체 데이터로 부리 길이와 두께의 관계를 회귀로 확인해 봅시다

- `bill_depth_mm`(반응변수)을 `bill_length_mm`(설명변수)으로 설명하는 **단순회귀**를 적합하세요.

- 기울기 계수와 p-값을 출력하세요.

- **이 결과만 보고했다면 어떤 결론이 되는지** 한 줄로 적으세요.

- `💡 힌트` `smf.ols(formula='...', data=bills).fit()` — 7장에서 쓴 방식과 같습니다.

- `📖 용어` **적합(fit)** — 모형의 **형태**만 정해 주면 절편과 기울기는 아직 빈칸입니다.
  그 빈칸을 데이터로부터 채워 넣는 것, 즉 흩어진 점들 사이로 **가장 잘 들어맞는 직선을 찾아내는 작업**이
  '적합'입니다. 코드에서는 `.fit()`이 그 순간이며, 메서드 이름 자체가 fit(적합)입니다.
  **"적합하세요" = "`.fit()`을 호출해 계수를 추정하세요"** 로 읽으면 됩니다.
  ('적용'과 다릅니다 — 적합이 먼저고, 적합해 둔 모형을 새 데이터에 쓰는 것이 적용입니다.)

In [ ]:
# 문제 4 · Q1
# 여기에 코드를 작성해주세요.

model_all = smf.ols(formula='bill_depth_mm ~ bill_length_mm', data=bills).fit()

slope_all = model_all.params['bill_length_mm']
pval_all = model_all.pvalues['bill_length_mm']

print("=== Q1. 전체 데이터 단순회귀 ===")
print("기울기 = %.4f,  p-값 = %.4g" % (slope_all, pval_all))
print("→ 이 결과만 보면: '부리가 길수록 부리는 얕아진다(음의 상관)'고 결론 내리게 됨.\n")


### Q2 · 종별로 층을 나눠 같은 회귀를 다시 해 봅시다

- 세 종(Adelie · Chinstrap · Gentoo) **각각에 대해** Q1과 똑같은 회귀를 적합하세요.

- 종별로 **n, 기울기 계수, p-값**을 표로 정리해 출력하세요.

- 이것은 강의 10장의 _"중학교 1·2·3학년으로 층을 나눈 후, 각 학년을 따로 해석"_ 과
  **똑같은 작업**입니다.

- `💡 힌트` `for 종, 그룹 in bills.groupby('species'):` 로 돌면서 각 그룹에 회귀를 적합하세요.

In [ ]:
# 문제 4 · Q2
# 여기에 코드를 작성해주세요.

print("=== Q2. 종별 층화 회귀 ===")
print("%10s %6s %12s %12s" % ("종", "n", "기울기", "p-값"))

strat_results = []
for species, group in bills.groupby('species'):
    m = smf.ols(formula='bill_depth_mm ~ bill_length_mm', data=group).fit()
    slope = m.params['bill_length_mm']
    p = m.pvalues['bill_length_mm']
    strat_results.append({'species': species, 'n': len(group), 'slope': slope, 'p': p})
    print("%10s %6d %12.4f %12.4g" % (species, len(group), slope, p))


### Q3 · 종을 회귀식에 함께 넣어 통제해 봅시다

- 이번엔 층을 나누지 않고, **회귀식에 '종'을 함께 넣어** 한 번에 적합하세요.

- Q1의 결과와 **기울기·p-값·R²를 나란히 출력**해 비교하세요.

- `💡 힌트` `'bill_depth_mm ~ bill_length_mm + C(species)'`
  — `C()`는 범주형 변수를 더미로 자동 변환해 줍니다.
  (주의: `C`라는 이름의 변수를 만들면 이 기능이 가려집니다)

In [ ]:
# 문제 4 · Q3
# 여기에 코드를 작성해주세요.

model_ctrl = smf.ols(formula='bill_depth_mm ~ bill_length_mm + C(species)', data=bills).fit()

slope_ctrl = model_ctrl.params['bill_length_mm']
pval_ctrl = model_ctrl.pvalues['bill_length_mm']

print("\n=== Q3. 종을 통제한 다중회귀 ===")
print("%20s %12s %12s %10s" % ("모델", "기울기", "p-값", "R²"))
print("%20s %12.4f %12.4g %10.4f" % ("Q1 (단순회귀)", slope_all, pval_all, model_all.rsquared))
print("%20s %12.4f %12.4g %10.4f" % ("Q3 (종 통제)", slope_ctrl, pval_ctrl, model_ctrl.rsquared))


### Q4 · 관찰연구에서 처치군과 대조군은 애초에 다릅니다

- 아래 데이터 준비 셀을 실행하면 학생 2,000명의 데이터 **두 벌**이 만들어집니다.
  같은 학생들인데 **강의 배정 방식만** 다릅니다.

- 먼저 **관찰연구(`df_obs`)**를 보세요. 성적이 좋고 열심인 학생이 스스로 신청한 상황입니다.

- ① 수강생과 비수강생의 `final_score` **단순 평균 차이**를 계산하세요.

- ② 두 집단의 **교란요인 평균**(`pre_score`, `study_hours`)도 함께 비교하세요.

- 참 효과는 **5.0점**입니다. 단순 평균 차이가 이 값과 얼마나 다른지 확인하세요.

- `💡 힌트` `df_obs.groupby('course')[['pre_score', 'study_hours', 'final_score']].mean()`

In [ ]:
# 문제 4 · 데이터 준비 (2/2) — 실행만 하세요
# 같은 학생 2,000명에게 '새 온라인 강의'를 배정하는 두 가지 방식
TRUE_EFFECT = 5.0                      # 강의의 진짜 효과: 기말고사 +5점
rng = np.random.default_rng(279)
n = 2000

pre_score   = rng.normal(60, 10, n)    # 교란요인 ① 사전 성적
study_hours = rng.normal(10,  3, n)    # 교란요인 ② 주당 학습시간

# 잠재결과 — 두 교란요인은 결과(기말고사 성적)에도 직접 영향을 줍니다
Y0 = 20 + 0.7 * pre_score + 0.8 * study_hours + rng.normal(0, 5, n)  # 강의를 안 들었을 때
Y1 = Y0 + TRUE_EFFECT                                                 # 강의를 들었을 때

# (A) 관찰연구 — 성적 좋고 열심인 학생이 스스로 신청 (선택 편향)
signup_logit = -0.6 + 0.10 * (pre_score - 60) + 0.20 * (study_hours - 10)
course_obs = rng.binomial(1, 1 / (1 + np.exp(-signup_logit)))

# (B) 무작위 통제 실험 — 동전을 던져 배정
course_rct = rng.binomial(1, 0.5, n)

df_obs = pd.DataFrame({'pre_score': pre_score, 'study_hours': study_hours,
                       'course': course_obs, 'final_score': np.where(course_obs == 1, Y1, Y0)})
df_rct = pd.DataFrame({'pre_score': pre_score, 'study_hours': study_hours,
                       'course': course_rct, 'final_score': np.where(course_rct == 1, Y1, Y0)})

print("참 효과(우리가 추정해야 하는 값): %.1f점\n" % TRUE_EFFECT)
print("관찰연구 df_obs — 수강생 %d명 (%.1f%%)" % (df_obs['course'].sum(), df_obs['course'].mean()*100))
print("RCT      df_rct — 수강생 %d명 (%.1f%%)" % (df_rct['course'].sum(), df_rct['course'].mean()*100))

In [ ]:
# 문제 4 · Q4
# 여기에 코드를 작성해주세요.

print("\n=== Q4. 관찰연구 (df_obs) ===")
obs_group_means = df_obs.groupby('course')[['pre_score', 'study_hours', 'final_score']].mean()
print(obs_group_means.round(3))

obs_naive_diff = obs_group_means.loc[1, 'final_score'] - obs_group_means.loc[0, 'final_score']
print("\n수강생 - 비수강생 final_score 단순 평균 차이: %.2f점" % obs_naive_diff)
print("참 효과: %.1f점  →  차이: %.2f점" % (5.0, obs_naive_diff - 5.0))

### Q5 · 같은 학생들을 동전 던지기로 배정하면 어떻게 될까요

- 이제 **RCT(`df_rct`)**에 대해 Q4와 **똑같은 두 가지 계산**을 하세요.

- 단순 평균 차이를 참 효과 5.0점과 비교하세요.

- 두 방식의 **교란요인 균형**을 하나의 표로 정리하면 대비가 선명해집니다.
  (행: `pre_score`, `study_hours` / 열: 관찰연구 차이, RCT 차이)

- `💡 힌트` Q4의 코드를 `df_rct`에 대해 반복하면 됩니다.
  표는 `pd.DataFrame({'관찰연구': [...], 'RCT': [...]}, index=[...])` 형태로 만들 수 있습니다.

In [ ]:
# 문제 4 · Q5
# 여기에 코드를 작성해주세요.

print("\n=== Q5. RCT (df_rct) ===")
rct_group_means = df_rct.groupby('course')[['pre_score', 'study_hours', 'final_score']].mean()
print(rct_group_means.round(3))

rct_naive_diff = rct_group_means.loc[1, 'final_score'] - rct_group_means.loc[0, 'final_score']
print("\n수강생 - 비수강생 final_score 단순 평균 차이: %.2f점" % rct_naive_diff)
print("참 효과: %.1f점  →  차이: %.2f점" % (5.0, rct_naive_diff - 5.0))

# 교란요인 균형 비교표
balance_table = pd.DataFrame({
    '관찰연구 차이': [
        obs_group_means.loc[1, 'pre_score'] - obs_group_means.loc[0, 'pre_score'],
        obs_group_means.loc[1, 'study_hours'] - obs_group_means.loc[0, 'study_hours'],
    ],
    'RCT 차이': [
        rct_group_means.loc[1, 'pre_score'] - rct_group_means.loc[0, 'pre_score'],
        rct_group_means.loc[1, 'study_hours'] - rct_group_means.loc[0, 'study_hours'],
    ]
}, index=['pre_score', 'study_hours'])

print("\n=== 교란요인 균형 비교 (수강생 - 비수강생) ===")
print(balance_table.round(3))

### 💬 정리 · 결과를 말로 설명해 보기

- Q1의 기울기는 **음수**인데 Q2·Q3의 기울기는 **양수**였습니다. 같은 데이터에서 부호가 뒤집힌
  이유는 무엇인가요? 이 현상의 이름은 무엇인가요?

- Q2(층별 분석)와 Q3(다중회귀)는 같은 결론을 줬습니다. **두 방법의 차이**는 무엇인가요?
  층이 아주 많아지면 어느 쪽이 유리할까요?

- Q4에서 관찰연구의 단순 비교값은 참 효과 5점과 크게 달랐습니다.
  **교란요인 균형 표를 근거로** 그 이유를 설명해 보세요.

- Q5의 RCT는 `pre_score`도 `study_hours`도 **통제하지 않았는데** 참값에 가까웠습니다.
  어떻게 이런 일이 가능한가요?

- 통제(Q3)와 무작위화(Q5)의 **결정적 차이**는 무엇인가요?
  **우리가 아직 모르는 교란요인**이 있다면 어느 쪽이 안전한가요?

- 그런데 현실에서 RCT가 늘 가능한가요? **불가능한 예를 두 개** 들어 보세요.

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 강의 10장은 RCT를 이렇게 설명했습니다 — _"중첩요인을 확인하지 않더라도, 그 효과를 무작위를
  이용하여 무효화할 수 있으므로, 알고자 하는 변수의 효과만 추정 가능합니다."_
  Q5의 균형 표가 이 문장의 증거입니다.

- **통제는 이름을 아는 교란요인만 지웁니다.** Q3에서 우리가 지운 것은 '종' 하나뿐입니다.
  성별·서식지·측정 연도는 그대로 남아 있습니다. 그 목록이 완전하다고 누가 보장해 줄까요?

- **무작위화는 아직 이름도 모르는 교란요인까지** 두 집단에 고르게 흩뿌립니다.
  이것이 RCT를 '인과추론의 황금 표준'이라 부르는 이유입니다.

- Q2 vs Q3 — 층별 분석은 층마다 따로 결론을 주므로 층이 많아지면 각 층의 n이 작아집니다.
  다중회귀는 모든 데이터를 한 번에 쓰지만, 대신 "층마다 기울기가 같다"는 가정을 덧붙입니다.

- 마지막 질문 — 강의는 담배와 건강을 예로 듭니다. 무작위로 흡연 집단을 만들 수 있을까요?
  윤리·비용·시간이 길을 막는 이 지점에서 **다음 문제가 시작됩니다.**

</details>

In [ ]:
# 문제 4 · 정리
# 여기에 의견을 작성해주세요.

# '종'이라는 변수가 부리 길이와 부리 두께 모두에 영향을 주는 교란요인인데, Q1에서는 이를 무시하고 세 종을 한데 섞어 전체 데이터로 회귀했기 때문입니다. Gentoo는 부리가 길고 얕은 반면 Adelie·Chinstrap은 상대적으로 짧고 깊어서, 종 내부에서는 "길수록 깊다(양의 관계)"인데도 종 간 차이가 이를 압도해 전체적으로는 "길수록 얕다(음의 관계)"처럼 보이게 됩니다. 이렇게 하위집단 내 경향과 전체 집계 경향이 반대로 나타나는 현상을 **심슨의 역설(Simpson's Paradox)**이라고 합니다.

# 층별 분석은 각 종마다 완전히 별도의 회귀선(절편과 기울기 모두 다르게)을 적합하는 반면, 다중회귀는 절편만 종별로 다르게 하고 기울기는 하나로 고정해 추정합니다(교호작용을 넣지 않는 한). 층이 아주 많아지면(예: 범주가 수십~수백 개) 층별 분석은 층마다 표본이 부족해져 추정이 불안정해지지만, 다중회귀는 데이터를 통합해서 쓰기 때문에 더 안정적으로 추정할 수 있어 유리합니다.

# 교란요인 균형 표를 보면 수강생 집단이 비수강생 집단보다 pre_score와 study_hours 평균이 더 높게 나타납니다. 즉 애초에 성적이 좋고 열심히 공부하는 학생이 스스로 강의를 신청했기 때문에, 수강생과 비수강생의 final_score 차이에는 강의의 진짜 효과(5점)뿐 아니라 이 두 교란요인 차이로 인한 격차까지 섞여 들어가 참값보다 크게 부풀려진 것입니다.

# 동전 던지기로 무작위 배정을 하면, 표본이 충분히 클 때 pre_score와 study_hours가 수강 여부와 무관하게 두 집단에 평균적으로 비슷하게 나뉩니다(교란요인 균형 표에서 두 값의 차이가 거의 0에 가까움). 두 집단이 교란요인 면에서 거의 동일하므로, 남은 차이는 순수하게 강의 효과만 반영하게 되어 통제 없이도 참값에 가까운 추정이 가능해집니다.

# 통제(회귀에 변수 추가)는 "내가 측정하고 생각해낸 교란요인"만 보정할 수 있는 반면, 무작위화는 측정했든 안 했든, 심지어 존재하는지도 몰랐던 교란요인까지 확률적으로 양쪽 집단에 고르게 분산시킵니다. 따라서 우리가 아직 모르는 교란요인이 있을 가능성을 생각하면 무작위화(RCT) 쪽이 훨씬 안전합니다.

# 아닙니다. 예를 들어 "흡연이 폐암에 미치는 영향"을 보려고 사람들에게 무작위로 흡연을 배정할 수는 없습니다(윤리적으로 해를 끼치는 처치를 강제할 수 없음). 또한 "부모의 소득 수준이 자녀의 학업 성취에 미치는 영향"처럼 애초에 개인의 소득이나 가정환경을 실험자가 무작위로 바꿔서 배정하는 것 자체가 현실적으로 불가능한 경우도 있습니다.

## 문제 5 · 미니 프로젝트 — 실험할 수 없을 때

`난이도 상` · `예상 40분`

**📖 상황**

- 문제 4에서 확인했습니다. 무작위 배정만 되면 인과효과는 **단순 평균 차이**로 구해집니다.
  문제는 **대부분의 현실에서 무작위 배정이 불가능**하다는 것입니다.

- 그래서 관찰 데이터만 놓고 인과효과에 다가가는 방법들이 있습니다. 강의 10.3의 두 가지를 씁니다.

- **① 경향점수 짝짓기(PSM)** — 처치를 받을 '경향'이 비슷한 사람끼리 짝지어
  **'통계적 쌍둥이'** 집단을 만듭니다.

- **② 이중차분법(DiD)** — 시간축을 도입해, 통제군의 변화를 '정책이 없었다면 일어났을 변화'로
  삼고 실험군의 변화에서 빼냅니다.

- 두 방법 모두 **가정 위에 서 있습니다.** 그래서 이 문제의 진짜 목표는 추정값을 구하는 것이 아니라,
  **그 가정을 검증하는 절차까지 해 보는 것**입니다.

**🎯 이 문제로 배우는 것**

- PSM으로 선택편향을 보정하고 **공변량 균형**으로 검증하며,
  DiD로 정책효과를 추정하고 **평행추세 확인·플라시보 검정**으로 반증을 시도합니다.

In [ ]:
# 문제 5 · 데이터 준비 (1/2) — 실행만 하세요
# 문제 4에서 만든 관찰연구 데이터(df_obs)를 그대로 이어서 씁니다.
print("관찰연구 데이터 df_obs: %d행" % len(df_obs))
print("  수강생 %d명 / 비수강생 %d명" % ((df_obs['course']==1).sum(), (df_obs['course']==0).sum()))
print("  단순 평균 비교(문제 4 Q4): %+.2f점" % (
    df_obs.loc[df_obs['course']==1, 'final_score'].mean()
    - df_obs.loc[df_obs['course']==0, 'final_score'].mean()))
print("  우리가 되찾아야 하는 참 효과: %+.1f점" % TRUE_EFFECT)

### Q1 · 경향점수를 추정해 봅시다

- **로지스틱 회귀**로 각 학생이 강의를 신청할 확률(= 경향점수)을 추정하세요.
  설명변수는 `pre_score`, `study_hours`이고 반응변수는 `course`입니다.

- 추정한 경향점수를 `df_obs['ps']` 열에 저장하세요. **Q2·Q3에서 계속 씁니다.**

- 수강생과 비수강생의 **경향점수 분포를 히스토그램으로 겹쳐 그려**,
  두 집단이 겹치는 구간이 있는지 확인하세요. (겹치는 구간이 있어야 짝을 찾을 수 있습니다)

- `💡 힌트` `sm.Logit(y, sm.add_constant(X)).fit(disp=0)` 으로 적합하고 `.predict()`로 확률을 얻습니다.
  히스토그램은 `plt.hist(..., alpha=0.5, label=...)`를 두 번 호출하면 겹쳐 그려집니다.

In [ ]:
# 문제 5 · Q1
# 여기에 코드를 작성해주세요.
X = sm.add_constant(df_obs[['pre_score', 'study_hours']])
y = df_obs['course']

logit_model = sm.Logit(y, X).fit(disp=0)
df_obs['ps'] = logit_model.predict(X)

print("=== Q1. 경향점수 모형 요약 ===")
print(logit_model.summary())

plt.figure(figsize=(7, 4.5))
plt.hist(df_obs.loc[df_obs['course']==1, 'ps'], bins=30, alpha=0.5, label='수강생', color='steelblue')
plt.hist(df_obs.loc[df_obs['course']==0, 'ps'], bins=30, alpha=0.5, label='비수강생', color='darkorange')
plt.xlabel("경향점수 (수강 확률)")
plt.ylabel("빈도")
plt.title("경향점수 분포 - 수강생 vs 비수강생")
plt.legend()
plt.tight_layout()
plt.show()

### Q2 · 1:1 짝짓기로 ATT를 추정해 봅시다

- 수강생(실험군) 한 명마다, **경향점수가 가장 가까운 비수강생(대조군) 1명**을 찾아 짝지으세요.

- 짝지어진 두 집단의 `final_score` 평균 차이를 계산하세요.
  이것이 **ATT**(처치집단에 대한 평균 처치효과)입니다.

- 세 값을 **나란히 출력**해 비교하세요.
  **① 단순 평균 비교(문제 4) ② PSM 이후 ATT ③ 참 효과 5.0점**

- `💡 힌트` `NearestNeighbors(n_neighbors=1).fit(대조군[['ps']])` 로 학습한 뒤
  `.kneighbors(실험군[['ps']])`를 호출하면 `(거리, 인덱스)`를 돌려줍니다.
  인덱스로 대조군을 골라낼 때는 `대조군.iloc[indices.flatten()]`를 씁니다.

In [ ]:
# 문제 5 · Q2
# 여기에 코드를 작성해주세요..

treated = df_obs[df_obs['course'] == 1].reset_index(drop=True)
control = df_obs[df_obs['course'] == 0].reset_index(drop=True)

nn = NearestNeighbors(n_neighbors=1)
nn.fit(control[['ps']])
distances, indices = nn.kneighbors(treated[['ps']])

matched_control = control.iloc[indices.flatten()].reset_index(drop=True)

att = treated['final_score'].mean() - matched_control['final_score'].mean()

naive_diff = (df_obs.loc[df_obs['course']==1, 'final_score'].mean()
              - df_obs.loc[df_obs['course']==0, 'final_score'].mean())

print("\n=== Q2. PSM 이후 ATT ===")
print("① 단순 평균 비교(문제 4): %+.2f점" % naive_diff)
print("② PSM 이후 ATT          : %+.2f점" % att)
print("③ 참 효과               : %+.1f점" % TRUE_EFFECT)

### Q3 · 짝짓기가 잘 됐는지 검증해 봅시다 — 공변량 균형

- 짝짓기는 **잘 됐다고 가정하면 안 되고, 확인해야 합니다.**
  이 확인 절차를 **공변량 균형(Covariate Balance)** 점검이라 부릅니다.

- `pre_score`와 `study_hours`에 대해, 두 집단의 평균 차이를
  **짝짓기 전 / 짝짓기 후**로 나누어 표로 만드세요.

- 짝짓기 후 차이가 0에 가까워졌다면 성공입니다.

- `💡 힌트` 표는 `pd.DataFrame({'짝짓기 전': [...], '짝짓기 후': [...]}, index=['pre_score', 'study_hours'])`

In [ ]:
# 문제 5 · Q3
# 여기에 코드를 작성해주세요.

before_pre  = (treated['pre_score'].mean() - control['pre_score'].mean())
before_hours = (treated['study_hours'].mean() - control['study_hours'].mean())

after_pre  = (treated['pre_score'].mean() - matched_control['pre_score'].mean())
after_hours = (treated['study_hours'].mean() - matched_control['study_hours'].mean())

balance_table = pd.DataFrame({
    '짝짓기 전': [before_pre, before_hours],
    '짝짓기 후': [after_pre, after_hours]
}, index=['pre_score', 'study_hours'])

print("\n=== Q3. 공변량 균형 (수강생 - 대조군, 짝짓기 전/후) ===")
print(balance_table.round(3))

### Q4 · DiD — 평행추세 가정을 **먼저** 확인해 봅시다

- 아래 데이터 준비 셀을 실행하면 서울·부산의 **2020~2025년 패널 데이터**가 만들어집니다.
  무료 공공 와이파이 정책은 **2024년 서울에만** 시행되었습니다.

- DiD를 계산하기 **전에** 가정을 확인합니다. **정책 시행 이전(2020~2023)만 잘라서**
  두 도시의 연도별 평균 데이터 사용량을 **꺾은선 그래프**로 그리세요.

- 연도별 **두 도시의 격차**(서울 − 부산)도 숫자로 출력하세요.

- 격차가 일정하게 유지되고 있다면, 평행추세 가정을 믿을 만합니다.

- `💡 힌트` `sns.lineplot(data=..., x='year', y='data_usage', hue='city', marker='o')`
  격차는 `.groupby(['year','city'])['data_usage'].mean().unstack()` 후 두 열을 빼면 구할 수 있습니다.

In [ ]:
# 문제 5 · 데이터 준비 (2/2) — 실행만 하세요
# 서울(실험군) / 부산(통제군)의 1인당 월 데이터 사용량(GB), 2020~2025년
POLICY_YEAR = 2024        # 서울시가 무료 공공 와이파이를 도입한 해
TRUE_DID    = 4.0         # 정책의 진짜 효과: +4 GB
rng = np.random.default_rng(2024)

records = []
for city, baseline, is_seoul in [('서울', 15.0, 1), ('부산', 10.0, 0)]:
    for person in range(300):
        person_effect = rng.normal(0, 2.0)                  # 사람마다 타고난 사용량 차이
        for year in range(2020, 2026):
            time_trend = 1.5 * (year - 2020)                # 두 도시에 똑같이 작용하는 시간 추세
            policy = TRUE_DID if (is_seoul and year >= POLICY_YEAR) else 0.0
            records.append((city, is_seoul, person + is_seoul * 300, year,
                            baseline + person_effect + time_trend + policy + rng.normal(0, 1.0)))

panel = pd.DataFrame(records, columns=['city', 'is_seoul', 'person_id', 'year', 'data_usage'])

print("패널 데이터: %d행 (도시 2 × 300명 × 6년)" % len(panel))
print("정책 시행: %d년, 서울만  /  참 효과: +%.1f GB\n" % (POLICY_YEAR, TRUE_DID))
print(panel.groupby(['year', 'city'])['data_usage'].mean().unstack().round(2))

In [ ]:
# 문제 5 · Q4
# 여기에 코드를 작성해주세요.

pre_policy = panel[panel['year'] < POLICY_YEAR]

plt.figure(figsize=(7, 4.5))
sns.lineplot(data=pre_policy, x='year', y='data_usage', hue='city', marker='o')
plt.title("정책 시행 이전 (2020~2023) 도시별 평균 데이터 사용량")
plt.xlabel("연도")
plt.ylabel("1인당 월 데이터 사용량 (GB)")
plt.tight_layout()
plt.show()

gap_table = pre_policy.groupby(['year', 'city'])['data_usage'].mean().unstack()
gap_table['격차(서울-부산)'] = gap_table['서울'] - gap_table['부산']

print("\n=== Q4. 정책 이전 연도별 두 도시 격차 ===")
print(gap_table.round(3))

### Q5 · 2×2 DiD 회귀로 정책효과를 추정해 봅시다

- 정책 **직전 해(2023)**와 **직후 해(2024)**만 사용합니다.

- `is_post`(2024년이면 1) 변수를 만들고, 다음 세 항이 들어간 회귀를 적합하세요.
  **`is_seoul` + `is_post` + 두 변수의 상호작용**

- 결과표를 출력하고, **계수 세 개가 각각 무엇을 재고 있는지** 주석에 적으세요.

- 상호작용항 계수를 참 효과 **+4.0 GB**와 비교하세요.

- `💡 힌트` `smf.ols('data_usage ~ is_seoul * is_post', data=...)`
  — 수식의 `*`는 두 주효과와 상호작용을 **한꺼번에** 넣어 줍니다.
  결과표에서 상호작용항의 이름은 `is_seoul:is_post`입니다.

In [ ]:
# 문제 5 · Q5
# 계수 세 개의 의미
# is_seoul         의 의미: 정책 이전(2023) 시점에서 서울과 부산의 기저 수준 차이
# is_post          의 의미: 정책과 무관하게 시간이 지나며 두 도시에 공통으로 나타난 변화(시간 추세)
# is_seoul:is_post 의 의미: 서울에서만, 정책 시행 이후에 추가로 발생한 변화 = DiD 추정치(정책효과)

# 여기에 코드를 작성해주세요.

did_data = panel[panel['year'].isin([2023, POLICY_YEAR])].copy()
did_data['is_post'] = (did_data['year'] == POLICY_YEAR).astype(int)

did_model = smf.ols('data_usage ~ is_seoul * is_post', data=did_data).fit()

print("\n=== Q5. 2×2 DiD 회귀 결과 (2023 vs 2024) ===")
print(did_model.summary())

did_coef = did_model.params['is_seoul:is_post']
did_p = did_model.pvalues['is_seoul:is_post']
print("\n상호작용항(정책효과) 계수: %.3f, p-값: %.4g" % (did_coef, did_p))
print("참 효과: +%.1f GB" % TRUE_DID)


### Q6 · 플라시보 검정으로 스스로 반증을 시도해 봅시다

- 우리 추정치를 **의심해 봅니다.** 정책이 **없었던 구간**에 똑같은 DiD를 적용하면
  효과가 0으로 나와야 정상입니다.

- **2022년 vs 2023년**(둘 다 정책 이전)으로 Q5와 똑같은 회귀를 돌리세요.

- 상호작용항 계수와 p-값을 출력하고, **검정을 통과했는지** 문장으로 판정하세요.

- 여기서 만약 유의한 효과가 나왔다면 **무엇을 의심해야 하는지**도 함께 적으세요.

- `💡 힌트` Q5의 코드에서 연도만 바꾸면 됩니다. `is_post`는 이제 2023년이면 1입니다.

In [ ]:
# 문제 5 · Q6
# 여기에 코드를 작성해주세요.

placebo_data = panel[panel['year'].isin([2022, 2023])].copy()
placebo_data['is_post'] = (placebo_data['year'] == 2023).astype(int)

placebo_model = smf.ols('data_usage ~ is_seoul * is_post', data=placebo_data).fit()

placebo_coef = placebo_model.params['is_seoul:is_post']
placebo_p = placebo_model.pvalues['is_seoul:is_post']

print("\n=== Q6. 플라시보 검정 (2022 vs 2023) ===")
print("상호작용항 계수: %.3f, p-값: %.4g" % (placebo_coef, placebo_p))

if placebo_p < 0.05:
    print("→ p < 0.05: 정책이 없었던 구간에서도 유의한 '효과'가 검출됨 → 검정 실패, 평행추세 가정 의심 필요")
else:
    print("→ p >= 0.05: 정책이 없었던 구간에서는 유의한 효과가 없음 → 플라시보 검정 통과, DiD 결과 신뢰도 강화")

### Q7 · 보고 문장으로 정리해 봅시다

- 지금까지의 결과를 종합해, 두 분석을 보고서에 어떻게 적을지 **각각 한 문장**으로 쓰세요.

- 숫자만 적지 말고 **어떤 가정 위에서 얻은 값인지, 어떻게 검증했는지**를 함께 담으세요.

In [ ]:
# 문제 5 · Q7
# PSM 보고 문장 (추정값 + 사용한 방법 + 균형 검증 결과 + 남은 한계): 경향점수(pre_score, study_hours로 추정)를 이용해 1:1 최근접 이웃 매칭을 수행한 결과, 온라인 강의 수강의 처치효과(ATT)는 약 X.X점으로 추정되었다(참고: 매칭 전 단순 비교는 이보다 크게 벗어나 있었음). 매칭 후 두 관측 교란요인의 평균 차이는 0에 가깝게 줄어들어 균형이 확인되었으나, 이 추정치는 관측된 교란요인만을 통제한 것으로, 측정하지 못한 변수(예: 학생의 학습 동기)가 남아 있다면 여전히 편향될 수 있다는 한계가 있다.
#
# DiD 보고 문장 (추정값 + 사용한 방법 + 평행추세 확인 + 플라시보 검정 결과): 서울시 무료 공공 와이파이 정책의 효과를 이중차분법(DiD)으로 추정한 결과, 정책 시행 전후 데이터 사용량 증가폭은 약 +X.X GB로 나타났다(참 효과 +4.0 GB와 유사). 정책 시행 이전 4개년(2020~2023) 동안 서울과 부산의 격차가 대체로 일정하게 유지되어 평행추세 가정이 성립함을 확인하였고, 정책이 없었던 구간(2022 vs 2023)에 동일한 방법을 적용한 플라시보 검정에서는 유의한 효과가 검출되지 않아 추정치의 신뢰도를 뒷받침하였다.

### 💬 정리 · 결과를 말로 설명해 보기

- Q2의 ATT는 참값 5점에 가까웠고, 문제 4의 단순 비교는 크게 벗어났습니다.
  **PSM이 한 일**을 한 문장으로 설명해 보세요.

- Q3의 균형 표에서 짝짓기 후 교란요인 차이는 어떻게 변했나요?
  만약 짝짓기 **후에도** 차이가 크게 남아 있었다면 어떻게 해야 할까요?

- PSM은 **경향점수 모형에 넣은 변수만** 균형을 맞춥니다. 만약 관측하지 못한 교란요인
  (예: 학생의 타고난 이해력)이 있다면, Q2의 ATT를 인과효과라고 부를 수 있을까요?
  **문제 4의 RCT와 비교**해서 답해 보세요.

- Q4에서 정책 이전 4년간 두 도시의 격차는 거의 일정했습니다. 이것이 DiD의 **어떤 가정**을
  지지하나요? 만약 이전부터 격차가 벌어지고 있었다면 Q5의 계수를 어떻게 읽어야 할까요?

- Q5의 상호작용항이 왜 정책효과인가요? 네 개 평균값을 직접 계산해
  `(A₂ − A₁) − (B₂ − B₁)`과 일치하는지 확인해 보세요.

- Q6의 플라시보 검정이 만약 **유의하게** 나왔다면, 우리는 무엇을 의심해야 하나요?

- PSM과 DiD는 각각 어떤 상황에 쓰는 도구인가요? 두 방법의 **데이터 요구조건**
  (단면 데이터 vs 패널 데이터)은 어떻게 다른가요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 강의 10장은 DiD의 인과효과를 이렇게 적었습니다 —
  _"인과 효과 = ΔA − ΔB = (A₂ − A₁) − (B₂ − B₁)"_.
  Q5의 상호작용항 계수를 이 식과 손으로 맞춰 보면 **소수점까지 정확히 일치**합니다.
  회귀는 이 뺄셈을 대신 해 주는 도구일 뿐입니다.

- 강의 10장은 평행추세에 대해 이렇게 덧붙였습니다 — _"이 가정이 맞는지 확인하려면, 정책이
  시행되기 이전 여러 시점의 데이터를 그래프로 그려보아야 합니다."_ Q4가 정확히 그 작업입니다.
  시점이 두 개뿐이라면 이 확인 자체가 **불가능**하다는 점에 주목하세요.

- 플라시보 검정의 논리는 **"효과가 없어야 할 곳에서 효과가 나오면 방법이 틀렸다"**입니다.
  자기 결론을 스스로 반증하려 드는 습관이 통계에서 이런 모양으로 나타납니다.
  통과했다고 해서 방법이 옳음이 증명된 것은 아니라는 점도 함께 기억해 두세요.

- 세 번째 질문이 이 실습 전체의 결론입니다 — **PSM·DiD는 RCT의 대체품이 아닙니다.**
  무작위화가 자동으로 해 주던 일을, **가정을 명시적으로 걸고** 대신 해내는 차선책입니다.
  그래서 "어떤 가정을 걸었는지" 말할 수 없다면 그 추정치는 쓸 수 없습니다.

- 마지막 질문 — 시간축이 없는 데이터에서는 어느 방법을 쓸 수 없나요?
  반대로, 처치 시점이 명확한 정책이라면 어느 쪽이 더 강력한가요?

- 이 문제에서 다루지 못한 준실험 방법으로 **회귀 불연속 설계(RDD)**가 있습니다.
  "커트라인 바로 위/아래는 거의 같은 사람들"이라는 발상을 쓰는 방법으로,
  기준점이 존재하는 정책(장학금 커트라인, 정년 등)에서 강력하게 작동합니다.

</details>

In [ ]:
# 문제 5 · 정리
# 여기에 의견을 작성해주세요.

# PSM이 한 일
# PSM은 관측된 교란요인(pre_score, study_hours)의 값이 비슷한 수강생-비수강생끼리 짝을 지어, 마치 "사전 조건이 같은 학생들을 비교하는 미니 실험"처럼 만들어줌으로써 선택 편향을 줄이고 참 효과에 가까운 추정치를 얻게 해줍니다.

# Q3의 균형 표 변화 / 차이가 남아있다면
# 짝짓기 후 pre_score와 study_hours의 평균 차이는 짝짓기 전보다 0에 훨씬 가까워집니다. 만약 짝짓기 후에도 차이가 크게 남아있다면, 경향점수 모형에 변수를 추가하거나(비선형항, 상호작용 등) 매칭 방법을 바꾸거나(캘리퍼 적용, 여러 명 매칭), 매칭이 잘 안 된 관측치를 제외하는 등의 조치가 필요합니다.

# 관측하지 못한 교란요인이 있다면 / RCT와 비교
# PSM은 경향점수 모형에 포함시킨 변수들만 균형을 맞춰주므로, 학생의 타고난 이해력처럼 관측되지 않은 교란요인이 존재하고 그것이 수강 여부와 성적 모두에 영향을 준다면 Q2의 ATT는 여전히 편향되어 있을 수 있어 엄밀한 "인과효과"라고 단정하기 어렵습니다. 반면 문제 4의 RCT는 무작위 배정을 통해 관측 여부와 무관하게 모든 교란요인(알려진 것이든 모르는 것이든)을 통계적으로 균형 맞춰주므로 인과효과 해석이 훨씬 안전합니다.

# Q4의 평행추세 / 격차가 벌어지고 있었다면
# 정책 이전 4년간 격차가 거의 일정했다는 것은 "정책이 없었다면 두 도시가 같은 추세로 나란히 움직였을 것"이라는 **평행추세 가정(parallel trends assumption)**을 지지하는 근거입니다. 만약 정책 이전부터 이미 격차가 점점 벌어지고 있었다면, Q5의 상호작용항 계수는 정책효과뿐 아니라 원래부터 존재하던 추세 차이까지 포함하고 있는 것이므로 순수한 정책효과로 해석할 수 없고 과대(혹은 과소)추정된 값으로 봐야 합니다.

# 상호작용항이 정책효과인 이유
# 상호작용항 계수는 "서울의 변화량(2024-2023)"에서 "부산의 변화량(2024-2023)"을 뺀 값, 즉 (A₂-A₁)-(B₂-B₁)과 정확히 같습니다. 부산의 변화량은 정책과 무관한 공통 시간 추세를 나타내므로, 이를 서울의 변화량에서 빼주면 시간 추세 효과가 상쇄되고 서울에서만 발생한 순수한 정책효과만 남게 됩니다. 네 집단 평균을 직접 계산해 이 식과 상호작용항 계수를 비교하면 정확히 일치하는 것을 확인할 수 있습니다.

# Q6의 플라시보 검정이 유의하게 나왔다면
# 정책이 시행되지 않은 구간에서도 유의한 "효과"가 검출되었다는 것은, 평행추세 가정이 애초에 성립하지 않았거나(두 도시가 원래도 다르게 움직이고 있었음), 두 도시에 서로 다르게 영향을 준 다른 사건(정책 외 요인)이 그 시기에 있었을 가능성을 의심해야 한다는 뜻입니다. 이 경우 Q5에서 얻은 DiD 추정치는 신뢰하기 어렵습니다.

# PSM과 DiD의 쓰임새 / 데이터 요구조건
# PSM은 한 시점의 단면 데이터(cross-sectional data)에서, 처치군과 대조군이 애초에 다른 특성을 가진 선택 편향 문제를 다룰 때 사용합니다. DiD는 처치 이전·이후 여러 시점을 관찰할 수 있는 패널(또는 반복 횡단면) 데이터가 필요하며, 시간에 따라 변하지 않는 관측되지 않은 차이(예: 도시 고유 특성)까지 상쇄할 수 있다는 장점이 있습니다. 즉 PSM은 "같은 시점, 다른 사람들"을 비교하는 도구이고, DiD는 "같은 대상(또는 집단), 다른 시점"의 변화를 비교하는 도구입니다.